In [ ]:
from scipy.interpolate import griddata
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd
import numpy as np
import re
from scipy import stats


### Estilo de los plots

In [ ]:
# ─────────────────────────────────────────────
# 1. Funciones de estilo
# ─────────────────────────────────────────────
def config_ax_state(ax):
    ax.tick_params(axis="both", which="both", direction="in", top=True, right=True)

def config_ax_IV(ax):
    # ax.grid(which="major", color="#DDDDDD", linewidth=0.8, zorder=-1)
    # ax.grid(which="minor", color="#DEDEDE", linestyle=":", linewidth=0.5, zorder=-1)
    ax.minorticks_on()
    # ax.tick_params(axis="both", which="both", direction="in", top=True, right=True)

def setup_paper_plt(plt, latex=True, scaling: float = 1):
    plt.rcParams.update(
        {
            "pgf.texsystem": "pdflatex",
            "text.usetex": latex,
            "font.family": "mathpazo",
            "text.latex.preamble": "\n".join(
                [
                    r"\usepackage[utf8]{inputenc}",
                    r"\usepackage[T1]{fontenc}",
                    r"\usepackage{siunitx}",
                    r"\usepackage{physics}",
                ]
            ),
        }
    )
    BIGGER_SIZE = 11 * scaling
    BIGGEST_SIZE = 14 * scaling
    plt.rc("font", size=BIGGER_SIZE)
    plt.rc("axes", titlesize=BIGGER_SIZE)
    plt.rc("axes", labelsize=BIGGEST_SIZE)
    plt.rc("xtick", labelsize=BIGGEST_SIZE)
    plt.rc("ytick", labelsize=BIGGEST_SIZE)
    plt.rc("legend", fontsize=BIGGER_SIZE)
    plt.rc("figure", titlesize=BIGGEST_SIZE)


setup_paper_plt(plt, latex=True, scaling=2.5)

### Lectura de datos

In [ ]:
# 2. Leemos los datos
# 2. Leemos los datos experimentales
nombre_archivo = "Datos_Barrido_Grosores_Tension.txt"
df = pd.read_csv(nombre_archivo, sep="\t")

x = df["Diámetro CF 1 (nm)"]
y = df["Diámetro CF 2 (nm)"]

# 3. Diccionarios de mapeo
mapa_creacion = {
    "Tensión Creación CF 1 (V)": r"$V_{S}$ CF 1 (V)",
    "Tensión Creación CF 2 (V)": r"$V_{S}$ CF 2 (V)",
}

mapa_rotura = {"Tensión rotura CF 1 (V)": r"$V_{RS}$ CF 1 (V)", "Tensión rotura 2 CF (V)": r"$V_{RS}$ CF 2 (V)"}

# Creamos la malla de altísima resolución común para todas las gráficas (200x200)
xi = np.linspace(0.25, 9.75, 200)
yi = np.linspace(0.25, 9.75, 200)
xi, yi = np.meshgrid(xi, yi)


### Voltaje con interpolación

In [ ]:
setup_paper_plt(plt, latex=True, scaling=2.5)

# ---------------------------------------------------------
# FIGURA 1: Tensiones de Creación
# ---------------------------------------------------------
fig1, axs1 = plt.subplots(1, 2, figsize=(20, 6))

# Calculamos vmin/vmax sobre las matrices ya interpoladas y simetrizadas
zi_matrices_creacion = []
for col_datos in mapa_creacion.keys():
    z = df[col_datos]
    zi = griddata((x, y), z, (xi, yi), method="linear")
    zi_simetrico = (zi + zi.T) / 2.0
    zi_matrices_creacion.append(zi_simetrico)

zi_matrices_creacion = np.abs(zi_matrices_creacion)

vmin_creacion = min(np.nanmin(m) for m in zi_matrices_creacion)
vmax_creacion = max(np.nanmax(m) for m in zi_matrices_creacion)
levels_creacion = np.linspace(vmin_creacion, vmax_creacion, 15)  # mismo array para ambos

print(f"Rango Creación: vmin={vmin_creacion:.2f}, vmax={vmax_creacion:.2f}")
print("Niveles Creación:", levels_creacion)

for ax, (col_datos, etiqueta_grafica), zi_simetrico in zip(axs1.flat, mapa_creacion.items(), zi_matrices_creacion):
    contour = ax.contourf(xi, yi, zi_simetrico, levels=levels_creacion, cmap="viridis")
    cbar = fig1.colorbar(contour, ax=ax)
    cbar.set_ticklabels([f"{v:.2f}" for v in levels_creacion])
    ax.set_xlabel("Diameter CF 1 (nm)")
    ax.set_ylabel("Diameter CF 2 (nm)")
    ax.set_title(etiqueta_grafica)
    ax.set_aspect("equal", adjustable="box")
    # ax.set_xlim(0.25, 9.75)
    # ax.set_ylim(0.25, 9.75)
    ax.set_xticks([0.25, 2.75, 5.25, 7.75, 9.75])
    ax.set_yticks([0.25, 2.75, 5.25, 7.75, 9.75])
    ax.set_xticklabels(["0.25", "2.75", "5.25", "7.75", "9.75"], rotation=0, ha="center")
    ax.set_yticklabels(["0.25", "2.75", "5.25", "7.75", "9.75"], rotation=0, va="center")
    ax.tick_params(axis="both", labelsize=32)
    ax.tick_params(axis="x", labelsize=32, pad=12)
    ax.set_aspect("equal", adjustable="box")
    config_ax_state(ax)

fig1.savefig("tensiones_creacion.pdf", dpi=300, bbox_inches="tight")
    
# ---------------------------------------------------------
# FIGURA 2: Tensiones de Rotura
# ---------------------------------------------------------
fig2, axs2 = plt.subplots(1, 2, figsize=(20, 6))

zi_matrices_rotura = []
for col_datos in mapa_rotura.keys():
    z = df[col_datos]
    zi = griddata((x, y), z, (xi, yi), method="linear")
    zi_simetrico = (zi + zi.T) / 2.0
    zi_matrices_rotura.append(zi_simetrico)

zi_matrices_rotura = np.abs(zi_matrices_rotura)
vmin_rotura = min(np.nanmin(m) for m in zi_matrices_rotura)
vmax_rotura = max(np.nanmax(m) for m in zi_matrices_rotura)
levels_rotura = np.linspace(vmin_rotura, vmax_rotura, 15)  # mismo array para ambos


for ax, (col_datos, etiqueta_grafica), zi_simetrico in zip(axs2.flat, mapa_rotura.items(), zi_matrices_rotura):
    ax.margins(0)
    contour = ax.contourf(
        xi, yi, zi_simetrico, levels=levels_rotura, cmap="viridis", vmin=vmin_rotura, vmax=vmax_rotura
    )
    cbar = fig2.colorbar(contour, ax=ax)
    cbar.set_ticklabels([f"{v:.2f}" for v in levels_rotura])
    ax.set_xlabel("Diameter CF 1 (nm)")
    ax.set_ylabel("Diameter CF 2 (nm)")
    ax.set_title(etiqueta_grafica)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(0.25, 9.75)
    ax.set_ylim(0.25, 9.75)
    ax.set_xticks([0.25, 2.75, 5.25, 7.75, 9.75])
    ax.set_yticks([0.25, 2.75, 5.25, 7.75, 9.75])
    ax.set_xticklabels(["0.25", "2.75", "5.25", "7.75", "9.75"], rotation=0, ha="center")
    ax.set_yticklabels(["0.25", "2.75", "5.25", "7.75", "9.75"], rotation=0, va="center")
    ax.tick_params(axis="both", labelsize=32)
    ax.tick_params(axis="x", labelsize=32, pad=12)
    ax.set_aspect("equal", adjustable="box")
    config_ax_state(ax)

fig2.savefig("tensiones_rotura.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# ---------------------------------------------------------
# FIGURA DIAGNÓSTICO: celdas medidas vs faltantes
# ---------------------------------------------------------
setup_paper_plt(plt, latex=True, scaling=2.5)
fig_diag, axs_diag = plt.subplots(1, 2, figsize=(17, 7))

diams = sorted(set(df["Diámetro CF 1 (nm)"]).union(set(df["Diámetro CF 2 (nm)"])))
n = len(diams)
idx = np.arange(n)

for ax, (col_datos, etiqueta_grafica) in zip(axs_diag.flat, mapa_creacion.items()):
    df_mat = df.pivot_table(
        index="Diámetro CF 2 (nm)", columns="Diámetro CF 1 (nm)", values=col_datos, aggfunc="mean"
    ).reindex(index=diams, columns=diams)

    Z = df_mat.values
    mask = np.isnan(Z)

    # Fondo rojo = celdas sin datos
    ax.pcolormesh(idx, idx, mask.astype(float), cmap="Greys", vmin=0, vmax=1)

    # Viridis = celdas con datos
    Z_masked = np.ma.array(Z, mask=mask)
    pcm = ax.pcolormesh(idx, idx, Z_masked, cmap="viridis")
    fig_diag.colorbar(pcm, ax=ax)
    

    ax.set_xticks(idx)
    ax.set_xticklabels([str(v) for v in diams], rotation=0, ha="right")
    ax.set_yticks(idx)
    ax.set_yticklabels([str(v) for v in diams])

    ax.set_xlabel("Diameter CF 1 (nm)")
    ax.set_ylabel("Diameter CF 2 (nm)")
    ax.set_title(etiqueta_grafica)
    ax.set_aspect("equal", adjustable="box")
    config_ax_state(ax)

fig_diag.tight_layout()
# fig_diag.savefig("diagnostico_datos.pdf", dpi=300, bbox_inches="tight")


In [ ]:
setup_paper_plt(plt, latex=True, scaling=2.5)

# ---------------------------------------------------------
# FIGURA 1: Tensiones de Creación
# ---------------------------------------------------------
fig1, axs1 = plt.subplots(1, 2, figsize=(20, 6))

# Calculamos vmin/vmax sobre las matrices ya interpoladas y simetrizadas
zi_matrices_creacion = []
for col_datos in mapa_creacion.keys():
    z = df[col_datos]
    zi = griddata((x, y), z, (xi, yi), method="linear")
    zi_simetrico = (zi + zi.T) / 2.0
    zi_matrices_creacion.append(zi_simetrico)

zi_matrices_creacion = np.abs(zi_matrices_creacion)

vmin_creacion = min(np.nanmin(m) for m in zi_matrices_creacion)
vmax_creacion = max(np.nanmax(m) for m in zi_matrices_creacion)
levels_creacion = np.linspace(vmin_creacion, vmax_creacion, 15)  # mismo array para ambos

print(f"Rango Creación: vmin={vmin_creacion:.2f}, vmax={vmax_creacion:.2f}")
print("Niveles Creación:", levels_creacion)

for ax, (col_datos, etiqueta_grafica), zi_simetrico in zip(axs1.flat, mapa_creacion.items(), zi_matrices_creacion):
    contour = ax.contourf(xi, yi, zi_simetrico, levels=levels_creacion, cmap="viridis")
    mask_unico = x >= y
    ax.scatter(x[mask_unico], y[mask_unico], color="white", s=40, zorder=5, linewidths=1, edgecolors="black")
    
    cbar = fig1.colorbar(contour, ax=ax)
    cbar.set_ticklabels([f"{v:.2f}" for v in levels_creacion])
    ax.set_xlabel("Diameter CF 1 (nm)")
    ax.set_ylabel("Diameter CF 2 (nm)")
    ax.set_title(etiqueta_grafica)
    ax.set_aspect("equal", adjustable="box")
    # ax.set_xlim(0.25, 9.75)
    # ax.set_ylim(0.25, 9.75)
    ax.set_xticks([0.25, 2.75, 5.25, 7.75, 9.75])
    ax.set_yticks([0.25, 2.75, 5.25, 7.75, 9.75])
    ax.set_xticklabels(["0.25", "2.75", "5.25", "7.75", "9.75"], rotation=0, ha="center")
    ax.set_yticklabels(["0.25", "2.75", "5.25", "7.75", "9.75"], rotation=0, va="center")
    ax.tick_params(axis="both", labelsize=32)
    ax.tick_params(axis="x", labelsize=32, pad=12)
    ax.set_aspect("equal", adjustable="box")
    config_ax_state(ax)

fig1.savefig("tensiones_creacion_dotted.pdf", dpi=300, bbox_inches="tight")

# ---------------------------------------------------------
# FIGURA 2: Tensiones de Rotura
# ---------------------------------------------------------
fig2, axs2 = plt.subplots(1, 2, figsize=(20, 6))

zi_matrices_rotura = []
for col_datos in mapa_rotura.keys():
    z = df[col_datos]
    zi = griddata((x, y), z, (xi, yi), method="linear")
    zi_simetrico = (zi + zi.T) / 2.0
    zi_matrices_rotura.append(zi_simetrico)

zi_matrices_rotura = np.abs(zi_matrices_rotura)

vmin_rotura = min(np.nanmin(m) for m in zi_matrices_rotura)
vmax_rotura = max(np.nanmax(m) for m in zi_matrices_rotura)
levels_rotura = np.linspace(vmin_rotura, vmax_rotura, 15)  # mismo array para ambos


for ax, (col_datos, etiqueta_grafica), zi_simetrico in zip(axs2.flat, mapa_rotura.items(), zi_matrices_rotura):
    ax.margins(0)
    contour = ax.contourf(
        xi, yi, zi_simetrico, levels=levels_rotura, cmap="viridis", vmin=vmin_rotura, vmax=vmax_rotura
    )
    mask_unico = x >= y
    ax.scatter(x[mask_unico], y[mask_unico], color="white", s=40, zorder=5, linewidths=1, edgecolors="black")
    
    cbar = fig2.colorbar(contour, ax=ax)
    cbar.set_ticklabels([f"{v:.2f}" for v in levels_rotura])
    ax.set_xlabel("Diameter CF 1 (nm)")
    ax.set_ylabel("Diameter CF 2 (nm)")
    ax.set_title(etiqueta_grafica)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(0.25, 9.75)
    ax.set_ylim(0.25, 9.75)
    ax.set_xticks([0.25, 2.75, 5.25, 7.75, 9.75])
    ax.set_yticks([0.25, 2.75, 5.25, 7.75, 9.75])
    ax.set_xticklabels(["0.25", "2.75", "5.25", "7.75", "9.75"], rotation=0, ha="center")
    ax.set_yticklabels(["0.25", "2.75", "5.25", "7.75", "9.75"], rotation=0, va="center")
    ax.tick_params(axis="both", labelsize=32)
    ax.tick_params(axis="x", labelsize=32, pad=12)
    ax.set_aspect("equal", adjustable="box")
    config_ax_state(ax)

fig2.savefig("tensiones_rotura_dotted.pdf", dpi=300, bbox_inches="tight")
plt.show()


### Temperatura SET

In [ ]:
setup_paper_plt(plt, latex=True, scaling=2.5)

nombre_archivo = "Datos_Barrido_Grosores_clean copy.txt"
df = pd.read_csv(nombre_archivo, sep="\t")

# Duplicar datos creando el triángulo inferior con CF1 y CF2 intercambiados
df_simetrico = df.copy()
df_simetrico_invertido = df.copy()

# Invertir las columnas de diámetros y temperaturas
df_simetrico_invertido = df_simetrico_invertido.rename(
    columns={
        "Diámetro CF 1 (nm)": "Diámetro CF 2 (nm)",
        "Diámetro CF 2 (nm)": "Diámetro CF 1 (nm)",
        "Temperatura máxima CF 1 (K)": "Temperatura máxima CF 2 (K)",
        "Temperatura máxima CF 2 (K)": "Temperatura máxima CF 1 (K)",
    }
)

# Combinar ambos dataframes (eliminar duplicados de la diagonal)
df_completo = pd.concat([df, df_simetrico_invertido], ignore_index=True)
df_completo = df_completo.drop_duplicates(subset=["Diámetro CF 1 (nm)", "Diámetro CF 2 (nm)"], keep="first")

print(f"Filas después de simetrizar: {len(df_completo)}")

# Ahora usar df_completo para interpolar
x = df_completo["Diámetro CF 1 (nm)"]
y = df_completo["Diámetro CF 2 (nm)"]

mapa_temp = {
    "Temperatura máxima CF 1 (K)": "CF 1 Max Temp. SET (K)",
    "Temperatura máxima CF 2 (K)": "CF 2 Max Temp. SET (K)",
}

xi = np.linspace(0.25, 9.75, 200)
yi = np.linspace(0.25, 9.75, 200)
xi, yi = np.meshgrid(xi, yi)

print(f"Filas totales: {len(df)}")
print(f"Temperaturas CF1 únicas: {df['Temperatura máxima CF 1 (K)'].nunique()}")
print(f"Temperaturas CF2 únicas: {df['Temperatura máxima CF 2 (K)'].nunique()}")

In [ ]:
# Pre-calculamos todas las matrices para obtener el rango global
zi_matrices_temp = []
for col_datos in mapa_temp.keys():
    z = df_completo[col_datos]
    zi = griddata((x, y), z, (xi, yi), method="linear")
    zi_matrices_temp.append(zi)  # Sin (zi + zi.T) / 2.0

vmin_temp = min(np.nanmin(m) for m in zi_matrices_temp)
vmax_temp = max(np.nanmax(m) for m in zi_matrices_temp)

print(f"Rango global de temperaturas: {vmin_temp:.2f} K a {vmax_temp:.2f} K")

levels_temp = np.linspace(vmin_temp, vmax_temp, 15)

fig, axs = plt.subplots(1, 2, figsize=(20, 6))

for ax, (col_datos, etiqueta_grafica), zi_simetrico in zip(axs.flat, mapa_temp.items(), zi_matrices_temp):
    ax.margins(0)
    contour = ax.contourf(xi, yi, zi_simetrico, levels=levels_temp, cmap="coolwarm")

    cbar = fig.colorbar(contour, ax=ax)
    cbar.set_ticklabels([f"{v:.0f}" for v in levels_temp])

    ax.set_xlabel("Diameter CF 1 (nm)")
    ax.set_ylabel("Diameter CF 2 (nm)")
    ax.set_title(etiqueta_grafica)

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(0.25, 9.75)
    ax.set_ylim(0.25, 9.75)
    ax.set_xticks([0.25, 2.75, 5.25, 7.75, 9.75])
    ax.set_yticks([0.25, 2.75, 5.25, 7.75, 9.75])
    ax.set_xticklabels(["0.25", "2.75", "5.25", "7.75", "9.75"], rotation=0, ha="center")
    ax.set_yticklabels(["0.25", "2.75", "5.25", "7.75", "9.75"], rotation=0, va="center")
    config_ax_state(ax)

for ax in axs.flat:
    ax.set_xlim(0.25, 9.75)
    ax.set_ylim(0.25, 9.75)
    ax.tick_params(axis="both", labelsize=30)
    ax.tick_params(axis="x", labelsize=30, pad=12)

fig.savefig("Temperaturas_SET_plot.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Pre-calculamos todas las matrices para obtener el rango global
zi_matrices_temp = []
for col_datos in mapa_temp.keys():
    z = df_completo[col_datos]
    zi = griddata((x, y), z, (xi, yi), method="linear")
    zi_matrices_temp.append(zi)  # Sin (zi + zi.T) / 2.0

vmin_temp = min(np.nanmin(m) for m in zi_matrices_temp)
vmax_temp = max(np.nanmax(m) for m in zi_matrices_temp)

vmin_temp = min(np.nanmin(m) for m in zi_matrices_temp)
vmax_temp = max(np.nanmax(m) for m in zi_matrices_temp)

print(f"Rango global de temperaturas: {vmin_temp:.2f} K a {vmax_temp:.2f} K")

levels_temp = np.linspace(vmin_temp, vmax_temp, 15)

fig, axs = plt.subplots(1, 2, figsize=(20, 6))

for ax, (col_datos, etiqueta_grafica), zi_simetrico in zip(axs.flat, mapa_temp.items(), zi_matrices_temp):
    ax.margins(0)
    contour = ax.contourf(xi, yi, zi_simetrico, levels=levels_temp, cmap="coolwarm")
    cbar = fig.colorbar(contour, ax=ax)
    cbar.set_ticklabels([f"{v:.0f}" for v in levels_temp])
    

    mask_unico = x >= y
    ax.scatter(x[mask_unico], y[mask_unico], color="white", s=40, zorder=5, linewidths=1, edgecolors="black")

    ax.set_xlabel("Diameter CF 1 (nm)")
    ax.set_ylabel("Diameter CF 2 (nm)")
    ax.set_title(etiqueta_grafica)

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(0.25, 9.75)
    ax.set_ylim(0.25, 9.75)
    ax.set_xticks([0.25, 2.75, 5.25, 7.75, 9.75])
    ax.set_yticks([0.25, 2.75, 5.25, 7.75, 9.75])
    ax.set_xticklabels(["0.25", "2.75", "5.25", "7.75", "9.75"], rotation=0, ha="center")
    ax.set_yticklabels(["0.25", "2.75", "5.25", "7.75", "9.75"], rotation=0, va="center")
    config_ax_state(ax)

for ax in axs.flat:
    ax.set_xlim(0.25, 9.75)
    ax.set_ylim(0.25, 9.75)
    ax.tick_params(axis="both", labelsize=30)
    ax.tick_params(axis="x", labelsize=30, pad=12)

fig.savefig("Temperaturas_SET_dotted.pdf", dpi=300, bbox_inches="tight")
plt.show()


### Temperaturas RESET

In [ ]:
setup_paper_plt(plt, latex=True, scaling=2.5)

nombre_archivo = "Datos_Barrido_Grosores_Temperatura_Reset.txt"
df = pd.read_csv(nombre_archivo, sep="\t")

# Duplicar datos creando el triángulo inferior con CF1 y CF2 intercambiados
df_simetrico = df.copy()
df_simetrico_invertido = df.copy()

# Invertir las columnas de diámetros y temperaturas
df_simetrico_invertido = df_simetrico_invertido.rename(
    columns={
        "Diámetro CF 1 (nm)": "Diámetro CF 2 (nm)",
        "Diámetro CF 2 (nm)": "Diámetro CF 1 (nm)",
        "Temperatura máxima CF 1 (K) RESET": "Temperatura máxima CF 2 (K) RESET",
        "Temperatura máxima CF 2 (K) RESET": "Temperatura máxima CF 1 (K) RESET",
    }
)

# Combinar ambos dataframes (eliminar duplicados de la diagonal)
df_completo = pd.concat([df, df_simetrico_invertido], ignore_index=True)
df_completo = df_completo.drop_duplicates(subset=["Diámetro CF 1 (nm)", "Diámetro CF 2 (nm)"], keep="first")

print(f"Filas después de simetrizar: {len(df_completo)}")

# Ahora usar df_completo para interpolar
x = df_completo["Diámetro CF 1 (nm)"]
y = df_completo["Diámetro CF 2 (nm)"]

mapa_temp = {
    "Temperatura máxima CF 1 (K) RESET": "CF 1 Max Temp. RESET (K)",
    "Temperatura máxima CF 2 (K) RESET": "CF 2 Max Temp. RESET (K)",
}

xi = np.linspace(0.25, 9.75, 200)
yi = np.linspace(0.25, 9.75, 200)
xi, yi = np.meshgrid(xi, yi)

print(f"Filas totales: {len(df)}")
print(f"Temperaturas CF1 únicas: {df['Temperatura máxima CF 1 (K) RESET'].nunique()}")
print(f"Temperaturas CF2 únicas: {df['Temperatura máxima CF 2 (K) RESET'].nunique()}")


In [ ]:
# Pre-calculamos todas las matrices para obtener el rango global
zi_matrices_temp = []
for col_datos in mapa_temp.keys():
    z = df_completo[col_datos]
    zi = griddata((x, y), z, (xi, yi), method="linear")
    zi_matrices_temp.append(zi)  # Sin (zi + zi.T) / 2.0
    

vmin_temp = min(np.nanmin(m) for m in zi_matrices_temp)
vmax_temp = max(np.nanmax(m) for m in zi_matrices_temp)

print(f"Rango global de temperaturas: {vmin_temp:.2f} K a {vmax_temp:.2f} K")

levels_temp = np.linspace(vmin_temp, vmax_temp, 15)

fig, axs = plt.subplots(1, 2, figsize=(20, 6))

for ax, (col_datos, etiqueta_grafica), zi_simetrico in zip(axs.flat, mapa_temp.items(), zi_matrices_temp):
    ax.margins(0)
    contour = ax.contourf(xi, yi, zi_simetrico, levels=levels_temp, cmap="coolwarm")

    cbar = fig.colorbar(contour, ax=ax)
    cbar.set_ticklabels([f"{v:.0f}" for v in levels_temp])

    ax.set_xlabel("Diameter CF 1 (nm)")
    ax.set_ylabel("Diameter CF 2 (nm)")
    ax.set_title(etiqueta_grafica)

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(0.25, 9.75)
    ax.set_ylim(0.25, 9.75)
    ax.set_xticks([0.25, 2.75, 5.25, 7.75, 9.75])
    ax.set_yticks([0.25, 2.75, 5.25, 7.75, 9.75])
    ax.set_xticklabels(["0.25", "2.75", "5.25", "7.75", "9.75"], rotation=0, ha="center")
    ax.set_yticklabels(["0.25", "2.75", "5.25", "7.75", "9.75"], rotation=0, va="center")
    config_ax_state(ax)

for ax in axs.flat:
    ax.set_xlim(0.25, 9.75)
    ax.set_ylim(0.25, 9.75)
    ax.tick_params(axis="both", labelsize=30)
    ax.tick_params(axis="x", labelsize=30, pad=12)

fig.savefig("temperaturas_RESET.pdf", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
# Pre-calculamos todas las matrices para obtener el rango global
zi_matrices_temp = []
for col_datos in mapa_temp.keys():
    z = df_completo[col_datos]
    zi = griddata((x, y), z, (xi, yi), method="linear")
    zi_matrices_temp.append(zi)  # Sin (zi + zi.T) / 2.0

vmin_temp = min(np.nanmin(m) for m in zi_matrices_temp)
vmax_temp = max(np.nanmax(m) for m in zi_matrices_temp)

vmin_temp = min(np.nanmin(m) for m in zi_matrices_temp)
vmax_temp = max(np.nanmax(m) for m in zi_matrices_temp)

print(f"Rango global de temperaturas: {vmin_temp:.2f} K a {vmax_temp:.2f} K")

levels_temp = np.linspace(vmin_temp, vmax_temp, 15)

fig, axs = plt.subplots(1, 2, figsize=(20, 6))

for ax, (col_datos, etiqueta_grafica), zi_simetrico in zip(axs.flat, mapa_temp.items(), zi_matrices_temp):
    ax.margins(0)
    contour = ax.contourf(xi, yi, zi_simetrico, levels=levels_temp, cmap="coolwarm")
    cbar = fig.colorbar(contour, ax=ax)
    cbar.set_ticklabels([f"{v:.0f}" for v in levels_temp])

    mask_unico = x >= y
    ax.scatter(x[mask_unico], y[mask_unico], color="white", s=40, zorder=5, linewidths=1, edgecolors="black")

    ax.set_xlabel("Diameter CF 1 (nm)")
    ax.set_ylabel("Diameter CF 2 (nm)")
    ax.set_title(etiqueta_grafica)

    ax.set_aspect("equal", adjustable="box")
    ax.set_xlim(0.25, 9.75)
    ax.set_ylim(0.25, 9.75)
    ax.set_xticks([0.25, 2.75, 5.25, 7.75, 9.75])
    ax.set_yticks([0.25, 2.75, 5.25, 7.75, 9.75])
    ax.set_xticklabels(["0.25", "2.75", "5.25", "7.75", "9.75"], rotation=0, ha="center")
    ax.set_yticklabels(["0.25", "2.75", "5.25", "7.75", "9.75"], rotation=0, va="center")
    config_ax_state(ax)

for ax in axs.flat:
    ax.set_xlim(0.25, 9.75)
    ax.set_ylim(0.25, 9.75)
    ax.tick_params(axis="both", labelsize=30)
    ax.tick_params(axis="x", labelsize=30, pad=12)

fig.savefig("temperaturas_RESET_dotted.pdf", dpi=300, bbox_inches="tight")
plt.show()


### Tensiones de SET y RESET

In [ ]:
setup_paper_plt(plt, latex=True, scaling=2.5)

# ─────────────────────────────────────────────
# 2. Cargar datos
# ─────────────────────────────────────────────
df = pd.read_csv("Datos_Barrido_Grosores_clean.txt", sep="\t")
df.columns = ["d_CF1", "d_CF2", "V_creat_CF1", "V_creat_CF2", "V_rot_CF1", "V_rot_CF2", "T_max_CF1", "T_max_CF2"]

diameters_CF1 = sorted(df["d_CF1"].unique())
diameters_CF2 = sorted(df["d_CF2"].unique())


print("Diámetros CF1:", diameters_CF1)
print("Diámetros CF2:", diameters_CF2)

_diameters_highlight = [1.75, 3.75, 5.75,7.75, 9.75]
# _diameters_highlight = [0.75, 1.75, 2.75, 3.75, 4.75, 5.75, 6.75, 7.75, 8.75, 9.75]

_palette = [
    "#e6194b",
    "#3cb44b",
    "#ffe119",
    "#4363d8",
    "#f58231",
    "#911eb4",
    "#42d4f4",
    "#f032e6",
    "#bfef45",
    "#fabed4",
    "#469990",
    "#dcbeff",
    "#9A6324",
    "#fffac8",
    "#800000",
]

colors_CF1 = {d: _palette[i % len(_palette)] for i, d in enumerate(_diameters_highlight)}
colors_CF2 = {d: _palette[i % len(_palette)] for i, d in enumerate(_diameters_highlight)}

In [ ]:

def plot_pair(ax_left, ax_right, col_left, col_right, ylabel_left, ylabel_right):

    # Subplot izquierdo: eje x = d_CF2, curvas = d_CF1 filtrado
    for d1 in _diameters_highlight:
        subset = df[df["d_CF1"] == d1].sort_values("d_CF2")
        ax_left.plot(
            subset["d_CF2"],
            subset[col_left],
            marker="o",
            markersize=6,
            linewidth=1.5,
            color=colors_CF1[d1],
            label=rf"$d_{{\rm CF1}} = {d1}\ \rm nm$",
        )
    ax_left.set_xlabel(r"$d_{\rm CF2}\ (\rm nm)$")
    ax_left.set_ylabel(ylabel_left)
    config_ax_IV(ax_left)
    ax_left.legend(
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
        frameon=True,
        framealpha=0.9,
        ncol=1,
        columnspacing=0.8,
        handlelength=1.5,
    )

    # Subplot derecho: eje x = d_CF1, curvas = d_CF2 filtrado
    for d2 in _diameters_highlight:
        subset = df[df["d_CF2"] == d2].sort_values("d_CF1")
        ax_right.plot(
            subset["d_CF1"],
            subset[col_right],
            marker="s",
            markersize=6,
            linewidth=1.5,
            color=colors_CF2[d2],
            label=rf"$d_{{\rm CF2}} = {d2}\ \rm nm$",
        )
    ax_right.set_xlabel(r"$d_{\rm CF1}\ (\rm nm)$")
    ax_right.set_ylabel(ylabel_right)
    config_ax_IV(ax_right)
    ax_right.legend(
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
        frameon=True,
        framealpha=0.9,
        ncol=1,
        columnspacing=0.8,
        handlelength=1.5,
    )

    # Escala Y compartida
    ymin = min(ax_left.get_ylim()[0], ax_right.get_ylim()[0])
    ymax = max(ax_left.get_ylim()[1], ax_right.get_ylim()[1])
    ax_left.set_ylim(ymin, ymax)
    ax_right.set_ylim(ymin, ymax)


# ─────────────────────────────────────────────
# 3. Figura CREACIÓN
# ─────────────────────────────────────────────
fig_creat, (ax_c1, ax_c2) = plt.subplots(1, 2, figsize=(28, 8))
plot_pair(
    ax_c1,
    ax_c2,
    col_left="V_creat_CF1",
    col_right="V_creat_CF2",
    ylabel_left=r"$V_{s\,\rm CF1}\ (\rm V)$",
    ylabel_right=r"$V_{s\,\rm CF2}\ (\rm V)$",
)
fig_creat.tight_layout()
fig_creat.savefig("plot_creacion_def.pdf", dpi=300, bbox_inches="tight")

# ─────────────────────────────────────────────
# 4. Figura ROTURA
# ─────────────────────────────────────────────
fig_rot, (ax_r1, ax_r2) = plt.subplots(1, 2, figsize=(28, 8))
plot_pair(
    ax_r1,
    ax_r2,
    col_left="V_rot_CF1",
    col_right="V_rot_CF2",
    ylabel_left=r"$V_{\rm rs,\,CF1}\ (\rm V)$",
    ylabel_right=r"$V_{\rm rs,\,CF2}\ (\rm V)$",
)
fig_rot.tight_layout()
fig_rot.savefig("plot_rotura_def.pdf", dpi=300, bbox_inches="tight")

# # ─────────────────────────────────────────────
# # 5. Figura TEMPERATURA MÁXIMA
# # ─────────────────────────────────────────────
# fig_temp, (ax_t1, ax_t2) = plt.subplots(1, 2, figsize=(28, 8))
# plot_pair(
#     ax_t1,
#     ax_t2,
#     col_left="T_max_CF1",
#     col_right="T_max_CF2",
#     ylabel_left=r"$T_{\rm max,\,CF1}\ (\rm K)$",
#     ylabel_right=r"$T_{\rm max,\,CF2}\ (\rm K)$",
# )
# fig_temp.tight_layout()
# fig_temp.savefig("plot_temperatura_def.pdf", dpi=300, bbox_inches="tight")

print("Plots guardados.")


In [ ]:
# ─────────────────────────────────────────────
# 2. Cargar datos
# ─────────────────────────────────────────────
df = pd.read_csv("Datos_Barrido_Grosores_clean.txt", sep="\t")
df.columns = ["d_CF1", "d_CF2", "V_creat_CF1", "V_creat_CF2", "V_rot_CF1", "V_rot_CF2", "T_max_CF1", "T_max_CF2"]

diameters_CF1 = sorted(df["d_CF1"].unique())
diameters_CF2 = sorted(df["d_CF2"].unique())

_diameters_highlight = [1.75, 3.75, 5.75, 7.75, 9.75]
# _diameters_highlight = [0.75, 1.75, 2.75, 3.75, 4.75, 5.75, 6.75, 7.75, 8.75, 9.75]

_palette = [
    "#e6194b",
    "#3cb44b",
    "#ffe119",
    "#4363d8",
    "#f58231",
    "#911eb4",
    "#42d4f4",
    "#f032e6",
    "#bfef45",
    "#fabed4",
    "#469990",
    "#dcbeff",
    "#9A6324",
    "#fffac8",
    "#800000",
]

colors_CF1 = {d: _palette[i % len(_palette)] for i, d in enumerate(_diameters_highlight)}
colors_CF2 = {d: _palette[i % len(_palette)] for i, d in enumerate(_diameters_highlight)}


def plot_pair_regression(ax_left, ax_right, col_left, col_right, ylabel_left, ylabel_right, show_data=False):

    x_fit = np.linspace(df["d_CF2"].min(), df["d_CF2"].max(), 200)

    # Subplot izquierdo: eje x = d_CF2, curvas = d_CF1 filtrado
    for d1 in _diameters_highlight:
        subset = df[df["d_CF1"] == d1].sort_values("d_CF2")
        x = subset["d_CF2"].values
        y = subset[col_left].values

        slope, intercept, r, _, _ = stats.linregress(x, y)

        if show_data:
            ax_left.scatter(x, y, s=40, color=colors_CF1[d1], alpha=1, zorder=2)

        ax_left.plot(
            x_fit,
            slope * x_fit + intercept,
            linewidth=2.5,
            color=colors_CF1[d1],
            label=rf"$d_{{\rm CF1}} = {d1}\ \rm nm$",
        )

    ax_left.set_xlabel(r"$d_{\rm CF2}\ (\rm nm)$")
    ax_left.set_ylabel(ylabel_left)
    config_ax_IV(ax_left)
    ax_left.legend(
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
        frameon=True,
        framealpha=0.9,
        ncol=1,
        columnspacing=0.8,
        handlelength=1.5,
    )

    x_fit = np.linspace(df["d_CF1"].min(), df["d_CF1"].max(), 200)

    # Subplot derecho: eje x = d_CF1, curvas = d_CF2 filtrado
    for d2 in _diameters_highlight:
        subset = df[df["d_CF2"] == d2].sort_values("d_CF1")
        x = subset["d_CF1"].values
        y = subset[col_right].values

        slope, intercept, r, _, _ = stats.linregress(x, y)

        if show_data:
            ax_right.scatter(x, y, s=40, color=colors_CF2[d2], alpha=1, zorder=2)

        ax_right.plot(
            x_fit,
            slope * x_fit + intercept,
            linewidth=2.5,
            color=colors_CF2[d2],
            label=rf"$d_{{\rm CF2}} = {d2}\ \rm nm$",
        )

    ax_right.set_xlabel(r"$d_{\rm CF1}\ (\rm nm)$")
    ax_right.set_ylabel(ylabel_right)
    config_ax_IV(ax_right)
    ax_right.legend(
        loc="center left",
        bbox_to_anchor=(1.01, 0.5),
        frameon=True,
        framealpha=0.9,
        ncol=1,
        columnspacing=0.8,
        handlelength=1.5,
    )

    # Escala Y compartida
    ymin = min(ax_left.get_ylim()[0], ax_right.get_ylim()[0])
    ymax = max(ax_left.get_ylim()[1], ax_right.get_ylim()[1])
    ax_left.set_ylim(ymin, ymax)
    ax_right.set_ylim(ymin, ymax)


# ─────────────────────────────────────────────
# Versión 1: solo regresión
# ─────────────────────────────────────────────
for name, col_l, col_r, yl, yr in [
    ("creacion", "V_creat_CF1", "V_creat_CF2", r"$V_{s\,\rm CF1}\ (\rm V)$", r"$V_{s\,\rm CF2}\ (\rm V)$"),
    ("rotura", "V_rot_CF1", "V_rot_CF2", r"$V_{\rm rs,\,CF1}\ (\rm V)$", r"$V_{\rm rs,\,CF2}\ (\rm V)$"),
]:
    fig, (axL, axR) = plt.subplots(1, 2, figsize=(28, 8))
    plot_pair_regression(axL, axR, col_l, col_r, yl, yr, show_data=False)
    fig.tight_layout()
    fig.savefig(f"plot_{name}_regression.pdf", dpi=300, bbox_inches="tight")

# ─────────────────────────────────────────────
# Versión 2: regresión + datos
# ─────────────────────────────────────────────
for name, col_l, col_r, yl, yr in [
    ("creacion", "V_creat_CF1", "V_creat_CF2", r"$V_{s\,\rm CF1}\ (\rm V)$", r"$V_{s\,\rm CF2}\ (\rm V)$"),
    ("rotura", "V_rot_CF1", "V_rot_CF2", r"$V_{\rm rs,\,CF1}\ (\rm V)$", r"$V_{\rm rs,\,CF2}\ (\rm V)$"),
]:
    fig, (axL, axR) = plt.subplots(1, 2, figsize=(28, 8))
    plot_pair_regression(axL, axR, col_l, col_r, yl, yr, show_data=True)
    fig.tight_layout()
    fig.savefig(f"plot_{name}_regression_data.pdf", dpi=300, bbox_inches="tight")

print("Plots guardados.")


In [ ]:
_figures = [
    ("creacion", "V_creat_CF1", "V_creat_CF2"),
    ("rotura", "V_rot_CF1", "V_rot_CF2"),
]


# ─────────────────────────────────────────────
# 2. Estimar sigma por simetría (d1,d2) <-> (d2,d1)
# ─────────────────────────────────────────────
def get_sigma(df, col, d_fixed, filter_col, x_col):
    """
    Para cada fila con filter_col == d_fixed, busca su transpuesta
    (filter_col y x_col intercambiados) y calcula sigma a partir
    de las diferencias entre pares simétricos.
    """
    subset = df[df[filter_col] == d_fixed].sort_values(x_col)
    diffs = []
    for _, row in subset.iterrows():
        d_other = row[x_col]
        # fila transpuesta: filter_col y x_col intercambiados
        mirror = df[(df[filter_col] == d_other) & (df[x_col] == d_fixed)]
        if not mirror.empty:
            diff = row[col] - mirror.iloc[0][col]
            diffs.append(diff)
    if len(diffs) < 2:
        return None
    # sigma por propagación: Var(A-B) = Var(A) + Var(B) = 2*sigma² → sigma = std/sqrt(2)
    return np.std(diffs, ddof=1) / np.sqrt(2)


# ─────────────────────────────────────────────
# 3. Nivel de significancia mínimo (bajando desde 0.80)
# ─────────────────────────────────────────────
def chi2_min_alpha(p_value, alpha_start=0.80, alpha_step=0.05):
    for alpha in np.arange(alpha_start, 0.0 - alpha_step, -alpha_step):
        alpha = round(alpha, 10)
        if p_value >= alpha:
            return alpha
    return round(alpha_step, 10)


# ─────────────────────────────────────────────
# 4. Cálculo estadístico completo
# ─────────────────────────────────────────────
def compute_stats(x, y, sigma):
    n = len(x)
    dof = n - 2

    slope, intercept, r, _, _ = stats.linregress(x, y)

    # Errores estándar de pendiente y término independiente
    Sxx = np.sum((x - np.mean(x)) ** 2)
    slope_err = sigma / np.sqrt(Sxx)
    intercept_err = sigma * np.sqrt(np.sum(x**2) / (n * Sxx))

    # Chi² experimental con sigma independiente
    y_fit = slope * x + intercept
    residuals = y - y_fit
    chi2_exp = np.sum((residuals / sigma) ** 2)
    p_value = 1 - stats.chi2.cdf(chi2_exp, dof)

    # Chi² teórico al nivel de significancia mínimo >= 0.80
    alpha_min = chi2_min_alpha(p_value, alpha_start=0.80, alpha_step=0.05)
    chi2_theo = stats.chi2.ppf(1 - alpha_min, dof)

    return {
        "slope": slope,
        "slope_err": slope_err,
        "intercept": intercept,
        "intercept_err": intercept_err,
        "chi2_exp": chi2_exp,
        "chi2_theo": chi2_theo,
        "alpha_min": alpha_min,
        "p_value": p_value,
        "R2": r**2,
        "dof": dof,
        "PASS": chi2_exp <= chi2_theo,
    }


# ─────────────────────────────────────────────
# 5. Generar tabla
# ─────────────────────────────────────────────
rows = []

for name, col_l, col_r in _figures:
    for d in _diameters_highlight:
        # CF1: eje x = d_CF2, filtro d_CF1 == d
        subset = df[df["d_CF1"] == d].sort_values("d_CF2")
        x = subset["d_CF2"].values
        y = subset[col_l].values
        sigma = get_sigma(df, col_l, d, filter_col="d_CF1", x_col="d_CF2")
        if sigma is not None and sigma > 0:
            s = compute_stats(x, y, sigma)
        else:
            s = {
                k: np.nan
                for k in [
                    "slope",
                    "slope_err",
                    "intercept",
                    "intercept_err",
                    "chi2_exp",
                    "chi2_theo",
                    "alpha_min",
                    "p_value",
                    "R2",
                    "dof",
                    "PASS",
                ]
            }
        rows.append(
            {
                "Magnitud": name,
                "Filamento": "CF1",
                "d_fijo (nm)": d,
                "eje_x": "d_CF2",
                "Pendiente": s["slope"],
                "Err_pendiente": s["slope_err"],
                "Termino_ind": s["intercept"],
                "Err_term_ind": s["intercept_err"],
                "chi2_exp": s["chi2_exp"],
                "chi2_teo": s["chi2_theo"],
                "alpha_min": s["alpha_min"],
                "p_value": s["p_value"],
                "R2": s["R2"],
                "dof": s["dof"],
                "PASS": s["PASS"],
            }
        )

        # CF2: eje x = d_CF1, filtro d_CF2 == d
        subset = df[df["d_CF2"] == d].sort_values("d_CF1")
        x = subset["d_CF1"].values
        y = subset[col_r].values
        sigma = get_sigma(df, col_r, d, filter_col="d_CF2", x_col="d_CF1")
        if sigma is not None and sigma > 0:
            s = compute_stats(x, y, sigma)
        else:
            s = {
                k: np.nan
                for k in [
                    "slope",
                    "slope_err",
                    "intercept",
                    "intercept_err",
                    "chi2_exp",
                    "chi2_theo",
                    "alpha_min",
                    "p_value",
                    "R2",
                    "dof",
                    "PASS",
                ]
            }
        rows.append(
            {
                "Magnitud": name,
                "Filamento": "CF2",
                "d_fijo (nm)": d,
                "eje_x": "d_CF1",
                "Pendiente": s["slope"],
                "Err_pendiente": s["slope_err"],
                "Termino_ind": s["intercept"],
                "Err_term_ind": s["intercept_err"],
                "chi2_exp": s["chi2_exp"],
                "chi2_teo": s["chi2_theo"],
                "alpha_min": s["alpha_min"],
                "p_value": s["p_value"],
                "R2": s["R2"],
                "dof": s["dof"],
                "PASS": s["PASS"],
            }
        )

stats_df = pd.DataFrame(rows)

for col in ["Pendiente", "Err_pendiente", "Termino_ind", "Err_term_ind", "chi2_exp", "chi2_teo", "p_value", "R2"]:
    stats_df[col] = stats_df[col].round(6)
stats_df["alpha_min"] = stats_df["alpha_min"].round(2)

stats_df.to_csv("estadisticos_regresion.csv", sep="\t", index=False)
print("Guardado en estadisticos_regresion.txt")
print(stats_df.to_string(index=False))


In [ ]:
_figures = [
    ("creacion", "V_creat_CF1", "V_creat_CF2"),
    ("rotura", "V_rot_CF1", "V_rot_CF2"),
]


# ─────────────────────────────────────────────
# 2. Cálculo estadístico
# ─────────────────────────────────────────────
def compute_stats(x, y):
    n = len(x)
    dof = n - 2

    slope, intercept, r, _, _ = stats.linregress(x, y)

    # Errores estándar de pendiente y término independiente
    y_fit = slope * x + intercept
    residuals = y - y_fit
    sigma = np.sqrt(np.sum(residuals**2) / dof)
    Sxx = np.sum((x - np.mean(x)) ** 2)
    slope_err = sigma / np.sqrt(Sxx)
    intercept_err = sigma * np.sqrt(np.sum(x**2) / (n * Sxx))

    return {
        "slope": slope,
        "slope_err": slope_err,
        "intercept": intercept,
        "intercept_err": intercept_err,
        "R2": r**2,
        "dof": dof,
    }


# ─────────────────────────────────────────────
# 3. Generar tabla
# ─────────────────────────────────────────────
rows = []

for name, col_l, col_r in _figures:
    for d in _diameters_highlight:
        # CF1: eje x = d_CF2, filtro d_CF1 == d
        subset = df[df["d_CF1"] == d].sort_values("d_CF2")
        s = compute_stats(subset["d_CF2"].values, subset[col_l].values)
        rows.append(
            {
                "Magnitud": name,
                "Filamento": "CF1",
                "d_fijo (nm)": d,
                "eje_x": "d_CF2",
                "Pendiente": s["slope"],
                "Err_pendiente": s["slope_err"],
                "Termino_ind": s["intercept"],
                "Err_term_ind": s["intercept_err"],
                "R2": s["R2"],
            }
        )

        # CF2: eje x = d_CF1, filtro d_CF2 == d
        subset = df[df["d_CF2"] == d].sort_values("d_CF1")
        s = compute_stats(subset["d_CF1"].values, subset[col_r].values)
        rows.append(
            {
                "Magnitud": name,
                "Filamento": "CF2",
                "d_fijo (nm)": d,
                "eje_x": "d_CF1",
                "Pendiente": s["slope"],
                "Err_pendiente": s["slope_err"],
                "Termino_ind": s["intercept"],
                "Err_term_ind": s["intercept_err"],
                "R2": s["R2"],
            }
        )

stats_df = pd.DataFrame(rows)

for col in ["Pendiente", "Err_pendiente", "Termino_ind", "Err_term_ind", "R2"]:
    stats_df[col] = stats_df[col].round(6)

stats_df.to_csv("estadisticos_regresion.txt", sep="\t", index=False)
print("Guardado en estadisticos_regresion.txt")
print(stats_df.to_string(index=False))


In [ ]:
import pandas as pd

df = pd.read_csv("Datos_Barrido_Grosores copy.txt", sep="\t")
df.columns = ["d_CF1", "d_CF2", "V_creat_CF1", "V_creat_CF2", "V_rot_CF1", "V_rot_CF2", "T_max_CF1", "T_max_CF2"]

pairs = df[["d_CF1", "d_CF2"]]
duplicated = pairs[pairs.duplicated(keep=False)]

if duplicated.empty:
    print("No hay pares repetidos.")
else:
    print(f"Pares (d_CF1, d_CF2) repetidos ({len(duplicated)} filas):\n")
    for _, row in duplicated.drop_duplicates().iterrows():
        count = ((df["d_CF1"] == row["d_CF1"]) & (df["d_CF2"] == row["d_CF2"])).sum()
        print(f"  d_CF1 = {row['d_CF1']} nm,  d_CF2 = {row['d_CF2']} nm  →  {count} veces")


In [ ]:
df = pd.read_csv("Datos_Barrido_Grosores copy.txt", sep="\t")
df.columns = ["d_CF1", "d_CF2", "V_creat_CF1", "V_creat_CF2", "V_rot_CF1", "V_rot_CF2", "T_max_CF1", "T_max_CF2"]

print(f"Filas antes: {len(df)}")

df_clean = df.drop_duplicates(subset=["d_CF1", "d_CF2"], keep="first")

print(f"Filas después: {len(df_clean)}")
print(f"Filas eliminadas: {len(df) - len(df_clean)}")

df_clean.to_csv("Datos_Barrido_Grosores_clean.txt", sep="\t", index=False)
print("Guardado en Datos_Barrido_Grosores_clean.txt")


In [ ]:
# ── Configura aquí ───────────────────────────────────────────────────────
n_save = 5  # número de simulación (= num_simulacion + 1)
paso = 0  # paso guardado
columna = 20  # columna del dispositivo (0-based, sin contar bordes)
# ─────────────────────────────────────────────────────────────────────────

ruta_npz = Path("Results") / f"simulation_{n_save}" / "reset" / f"Estado_pp_reset_sim_{n_save}_paso_{paso}.npz"

if not ruta_npz.is_file():
    raise FileNotFoundError(f"No se encontró: {ruta_npz}")

data = np.load(ruta_npz)
temperatura = data["temperatura"]  # shape (eje_x, eje_y + 2)

if temperatura.ndim != 2 or temperatura.size == 0:
    raise ValueError("temperatura no disponible en este paso (sin percolación).")

# columna del dispositivo → índice en la matriz (+1 por borde izquierdo)
T_perfil = temperatura[:, columna + 1]

print(f"Perfil de temperatura para sim {n_save}, paso {paso}, columna {columna}:")
print(T_perfil)
print(f"El máximo de temperatura en esta columna es: {T_perfil.max():.2f} K")

cfg = load_simulation_config(n_save - 1, init_data_dir=Path("Init_data"))
atom_nm = cfg.params.atom_size * 1e9  # m → nm
distancia = np.arange(len(T_perfil)) * atom_nm

fig, ax = plt.subplots(figsize=(16, 14))
ax.plot(distancia, T_perfil, lw=1.5, color="tomato")
ax.set_xlabel("Distancia (nm)")
ax.set_ylabel("Temperatura (K)")
ax.set_title(f"Perfil de temperatura — paso {paso}, columna {columna}")
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()
